<a href="https://colab.research.google.com/github/Elliotn46/27-13001506-DE-JS-Assesment/blob/main/seismic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install ipython-sql

import os
os.makedirs("database", exist_ok=True)

%load_ext sql
%sql sqlite:///database/seismic.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [8]:
%%sql
--Stations Table
CREATE TABLE IF NOT EXISTS stations (
    network TEXT NOT NULL,
    station TEXT NOT NULL,
    latitude REAL NOT NULL,
    longitude REAL NOT NULL,
    elevation REAL,
    PRIMARY KEY (network, station)
);

--Channels Table
CREATE TABLE IF NOT EXISTS channels (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    network TEXT NOT NULL,
    station TEXT NOT NULL,
    location TEXT NOT NULL,
    channel TEXT NOT NULL,
    sampling_rate REAL NOT NULL,
    FOREIGN KEY (network, station) REFERENCES stations (network, station) ON DELETE CASCADE,
    UNIQUE (network, station, location, channel)
);

--Waveform Headers / Metadata Table
CREATE TABLE IF NOT EXISTS waveforms (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    channel_id INTEGER NOT NULL,
    starttime TEXT NOT NULL,
    endtime TEXT NOT NULL,
    npts INTEGER NOT NULL,
    file_source TEXT NOT NULL,
    FOREIGN KEY (channel_id) REFERENCES channels (id) ON DELETE CASCADE
);

 * sqlite:///database/seismic.db
Done.
Done.
Done.


[]

In [ ]:
# Initialize SQLite database
db_path = "database/seismic.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Execute schema creation
cursor.execute('''CREATE TABLE IF NOT EXISTS stations (network TEXT, station TEXT, latitude REAL, longitude REAL, elevation REAL, PRIMARY KEY (network, station));''')
cursor.execute('''CREATE TABLE IF NOT EXISTS channels (id INTEGER PRIMARY KEY AUTOINCREMENT, network TEXT, station TEXT, location TEXT, channel TEXT, sampling_rate REAL);''')
conn.commit()

# Load mseed files
mseed_files = ["data/SEP01.mseed", "data/SEP02.mseed", "data/SEP03.mseed"]
iris_client = Client("IRIS")

# Track discovered stations to query IRIS once
discovered_stations = set()

for file_path in mseed_files:
    if os.path.exists(file_path):
        st = obspy.read(file_path)
        for tr in st:
            net, sta, loc, cha = tr.stats.network, tr.stats.station, tr.stats.location, tr.stats.channel
            discovered_stations.add((net, sta))

            # Insert channel metadata
            cursor.execute('''INSERT OR IGNORE INTO channels (network, station, location, channel, sampling_rate) VALUES (?, ?, ?, ?, ?)''',
                           (net, sta, loc, cha, tr.stats.sampling_rate))
        conn.commit()

print(f"Processed miniSEED files. Discovered stations from data: {discovered_stations}")